In [5]:
from valhalla import Actor, get_config
from shapely.geometry import LineString
import geopandas as gpd
import polyline


## Cullompton to Plymouth

In [2]:
# WGS84 coordinates (lon, lat)
cullompton = (-3.3924, 50.8588)
plymouth = (-4.1427, 50.3755)


In [3]:
config = get_config(
    tile_dir="C:/hsma_lambda_geographic/hsma_lambda_geographic/data/devon-260422_valhalla-modified-traffic_tiles",
    tile_extract="",
    verbose=True,
)

In [4]:
actor = Actor(config)

query = {
    "locations": [
        {"lat": cullompton[1], "lon": cullompton[0]},
        {"lat": plymouth[1], "lon": plymouth[0]},
    ],
    "costing": "auto",
}

route = actor.route(query)

In [5]:
route_coords = [
    (lon, lat)
    for lat, lon in polyline.decode(
        route["trip"]["legs"][0]["shape"],
        precision=6,
    )
]

route = gpd.GeoDataFrame(
    geometry=[LineString(route_coords)],
    crs="EPSG:4326",
).to_crs(3857)

In [6]:
route

,geometry
0,"LINESTRING (-377619.09 6596346.163, -377613.07..."


In [7]:
route.to_file("data/cullompton_plymouth.geojson")

## Repeat for Cullompton to Cardiff

### Generate new tiles

We only have tiles for Devon at present, so we need to merge some osm pbf files and rerun. 


In [1]:
import osmium
from pathlib import Path


input_files = [
    "data/devon-260729.osm.pbf",
    "data/gloucestershire-260729.osm.pbf",
    "data/bristol-260729.osm.pbf",
    "data/somerset-260729.osm.pbf",
    "data/wales-260729.osm.pbf"
]

abs_output = "C:/hsma_lambda_geographic/hsma_lambda_geographic/data/south_west_to_wales.osm.pbf"

output_path = Path(abs_output)

if output_path.exists():
    output_path.unlink()

writer = osmium.SimpleWriter(abs_output)

reader = osmium.MergeInputReader()

for filename in input_files:
    reader.add_file(filename)

reader.apply(
    writer,
    simplify=False
)

writer.close()

In [2]:
from lokigi.travel_utils import prepare_valhalla_network

prepare_valhalla_network(
    osm_path="data/south_west_to_wales.osm.pbf",
    output_dir="data/",
    output_name="south_west_to_wales-260729_valhalla.json"
)

Generating Valhalla config...
Building Valhalla tiles (this may take a while)...
Valhalla build complete


{'config_path': 'C:\\hsma_lambda_geographic\\hsma_lambda_geographic\\data\\south_west_to_wales-260729_valhalla.json.json',
 'tile_dir': 'C:\\hsma_lambda_geographic\\hsma_lambda_geographic\\data\\south_west_to_wales-260729_valhalla.json_tiles',
 'traffic_path': 'C:\\hsma_lambda_geographic\\hsma_lambda_geographic\\data\\south_west_to_wales-260729_valhalla.json_traffic.tar'}

### Routing

In [3]:
# WGS84 coordinates (lon, lat)
cullompton = (-3.3924, 50.8588)
cardiff = (-3.1746, 51.5230)

In [6]:
config = get_config(
    tile_dir="C:/hsma_lambda_geographic/hsma_lambda_geographic/data/south_west_to_wales-260729_valhalla.json_tiles",
    tile_extract="",
    verbose=True,
)

In [7]:
actor = Actor(config)

query = {
    "locations": [
        {"lat": cullompton[1], "lon": cullompton[0]},
        {"lat": cardiff[1], "lon": cardiff[0]},
    ],
    "costing": "auto",
}

route = actor.route(query)

In [8]:
route_coords = [
    (lon, lat)
    for lat, lon in polyline.decode(
        route["trip"]["legs"][0]["shape"],
        precision=6,
    )
]

route = gpd.GeoDataFrame(
    geometry=[LineString(route_coords)],
    crs="EPSG:4326",
).to_crs(3857)

In [9]:
route

,geometry
0,"LINESTRING (-377619.09 6596346.163, -377613.07..."


In [10]:
route.to_file("data/cullompton_cardiff.geojson")